# Test du pipeline sur un échantillon

In [1]:
import pandas as pd
from dotenv import load_dotenv  # pour ton .env avec PGHOST etc.

# Charger .env (PGHOST=localhost, PGPASSWORD=etudia...)
load_dotenv()

True

In [2]:
# 1. Charger tes chunks enrichis
df_chunks = pd.read_csv("../data/processed/parcoursup_2026_enriched_chunks.csv")
print(f"Chunks totaux: {len(df_chunks)}")
df_sample = df_chunks.sample(10, random_state=42)
print(f"Test sur: {len(df_sample)} chunks")

Chunks totaux: 29026
Test sur: 10 chunks


In [3]:
# 2. Créer les embeddings (e5-base)
from src.backend.vector_store import build_vector_store
df_vs = build_vector_store(df_sample)
print("df_vs créée:", df_vs.shape)
print("Embedding exemple:", len(df_vs["embedding"].iloc[0]))

/Users/maelioviau-vonfelt/Desktop/Masters/Epitech Data/Cours/M2/Mémoires/Consulting - StudAI/etudIA/env_etudia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9072.26it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


df_vs créée: (10, 11)
Embedding exemple: 768


In [6]:
# 3. Insérer en Postgres/pgvector
from src.backend.pgvector_store import upsert_chunks
upsert_chunks(df_vs)
print("✅ Upsert OK ! Vérifie pgAdmin.")

✅ Upsert OK ! Vérifie pgAdmin.


In [5]:
# 4. Vérif en base
from src.backend.pgvector_store import get_pg_connection
conn = get_pg_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM formations_chunks_vectors;")
print(f"Lignes en base: {cur.fetchone()[0]}")
cur.close()
conn.close()

Lignes en base: 10
